# **Pneumonia Detection using Computer Vision**

**AIM:**
- To train various models (CNN and transformer based) on patient's Chest X ray dataset and predict weather patient has pneumonia or not.
- Train and tune the models to get best result possible.
- Compare the models based on various metrics like Acuracy, Precession, F1, number of parameters, training time, etc.
- Deploy the best model on HuggingFace.

- Pneumonia is an inflammatory condition of the lung primarily affecting the small air sacs known as alveoli.
- Symptoms: combination of productive or dry cough, chest pain, fever, and difficulty in breathing. The severity of the condition is variable.
- Usually caused by infection with viruses or bacteria.

**Pneumonia causes lung tissue to swell (inflammation) and can cause fluid or pus in lungs.**

More about Pneumonia:
1. [Pneumonia causes, symptoms and treatment](https://my.clevelandclinic.org/health/diseases/4471-pneumonia)
2. [Chest X-Ray: Pneumonia Vs Normal](https://radiologyinplainenglish.com/chest-x-ray-pneumonia-vs-normal/)


<div style="display:flex; gap:10px;">
<img src="https://my.clevelandclinic.org/-/scassets/images/org/health/articles/4471-pneumonia-01" width="300">
<img src="https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Fwww.verywellhealth.com%2Fthmb%2FtpYTWdo0Sf2Cx_UqX58nrMhbFUY%3D%2F1500x0%2Ffilters%3Ano_upscale()%3Amax_bytes(150000)%3Astrip_icc()%2Fwhat-are-alveoli-2249043-01-94dfddd4dfe9488b8056d586824c7c36.png&f=1&nofb=1&ipt=66fa09d8082244ab4c0e096d2459bdfe745b31e3e8ddb7f62fe4fb38f90ab100" width="400">  
</div>

## 0. Install required libraries


In [14]:
!pip install torchinfo
!pip install torchmetrics
!pip install wandb

In [15]:
import os
import wandb
wandb.login()

True

## 1. Setup project directory

The project directory is set to folder: `MyDrive/Colab Notebooks/My Projects/Pneumonia Detection` in drive. For this the drive must be mounted.

**Folder Structure**

```
Pneumonia Detection/
│
├── data/
│   ├──chest_xray/
│      ├── train/
│      ├── test/
│      ├── val/
│
├── src/
|   |
│   ├── dataset.py
│   ├── transform.py
│   ├── dataloader.py
│   ├── predictions.py
│   ├── plots.py
│   ├── utils.py
|   |
│   ├── trainer/
│   |   ├── engine.py
│   |   ├── train.py
│   |   ├── train.py
│   │
│   ├── models/
│       ├── model.py
│       ├── architectures/
│            ├── resnet50.py
│            ├── densenet121.py
│            ├── efficientnet_b2.py
│            ├── vit_b_16.py
|
│
├── artifacts/
│   ├── models/
│   ├── results/
│
├── pneumonia_detection.ipynb
├── .gitignore
├── config.yaml
└── README.md
```

In [16]:
import torch

#set the device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cpu


In [ ]:
#mount the google drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import os
from pathlib import Path


# project directory path
project_dir = Path(f"/content/drive/MyDrive/Colab Notebooks/My Projects/Pneumonia Detection")

# Create the project directory if it does not exist
if not project_dir.is_dir():
    print(f"[INFO] Creating directory {project_dir}...")
    project_dir.mkdir(parents=True, exist_ok=True)

print(f"[INFO] Project directory is ready at: {project_dir}")

In [ ]:
#make project_dir as the default directory
%cd {project_dir}

# Verify current directory
!pwd

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_6368/836210423.py", line 2, in <cell line: 0>
    get_ipython().run_line_magic('cd', '{project_dir}')
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 2418, in run_line_magic
    result = fn(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^
  File "<decorator-gen-85>", line 2, in cd
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/magic.py", line 187, in <lambda>
    call = lambda f, *a, **k: f(*a, **k)
                              ^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/magics/osm.py", line 342, in cd
    oldcwd = os.getcwd()
             ^^^^^^^^^^^
OSError: [Errno 107] Transport endpoint is not connected

During handling of the above exception, another exception occurred:

Traceback (mo

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_6368/836210423.py", line 2, in <cell line: 0>
    get_ipython().run_line_magic('cd', '{project_dir}')
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 2418, in run_line_magic
    result = fn(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^
  File "<decorator-gen-85>", line 2, in cd
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/magic.py", line 187, in <lambda>
    call = lambda f, *a, **k: f(*a, **k)
                              ^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/magics/osm.py", line 342, in cd
    oldcwd = os.getcwd()
             ^^^^^^^^^^^
OSError: [Errno 107] Transport endpoint is not connected

During handling of the above exception, another exception occurred:

Traceback (mo

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_6368/836210423.py", line 2, in <cell line: 0>
    get_ipython().run_line_magic('cd', '{project_dir}')
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 2418, in run_line_magic
    result = fn(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^
  File "<decorator-gen-85>", line 2, in cd
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/magic.py", line 187, in <lambda>
    call = lambda f, *a, **k: f(*a, **k)
                              ^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/magics/osm.py", line 342, in cd
    oldcwd = os.getcwd()
             ^^^^^^^^^^^
OSError: [Errno 107] Transport endpoint is not connected

During handling of the above exception, another exception occurred:

Traceback (mo

In [ ]:
# creating necessery folders

(project_dir / "data").mkdir(parents=True, exist_ok=True)

(project_dir / "src").mkdir(parents=True, exist_ok=True)
(project_dir / "src/models/architectures").mkdir(parents=True, exist_ok=True)

(project_dir / "artifacts").mkdir(parents=True, exist_ok=True)
(project_dir / "artifacts/models").mkdir(parents=True, exist_ok=True)
(project_dir / "artifacts/results").mkdir(parents=True, exist_ok=True)

(project_dir / "src/trainer").mkdir(parents=True, exist_ok=True)

In [ ]:
from src.utils import set_seeds
set_seeds()

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_6368/3407768301.py", line 1, in <cell line: 0>
    from src.utils import set_seeds
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1322, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 1262, in _find_spec
  File "<frozen importlib._bootstrap_external>", line 1532, in find_spec
  File "<frozen importlib._bootstrap_external>", line 1504, in _get_spec
  File "<frozen importlib._bootstrap_external>", line 1483, in _path_importer_cache
OSError: [Errno 107] Transport endpoint is not connected

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 2099, i

## 2. Setup data directory and download the dataset

**Dataset Description**

1. **Dataset used:** [chest-xray-pneumonia (Kaggle)](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia)

2. **Dataset description:** The dataset is organized into 3 folders (train, test, val) and contains subfolders for each image category (Pneumonia/Normal). There are 5,863 X-Ray images (JPEG) and 2 categories (Pneumonia/Normal).

The dataset contains very less validation sets (16 total). So the validation set is merged with the train. And then the train is splited into $80 \% $ training and $20 \% $ validation splits.

### 2.1 Download the data

In [ ]:
from src.dataset import download_data

data_dir = download_data(target_dir=project_dir / "data")

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_6368/3685161403.py", line 1, in <cell line: 0>
    from src.dataset import download_data
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1322, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 1262, in _find_spec
  File "<frozen importlib._bootstrap_external>", line 1532, in find_spec
  File "<frozen importlib._bootstrap_external>", line 1504, in _get_spec
  File "<frozen importlib._bootstrap_external>", line 1483, in _path_importer_cache
OSError: [Errno 107] Transport endpoint is not connected

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 2

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_6368/3685161403.py", line 1, in <cell line: 0>
    from src.dataset import download_data
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1322, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 1262, in _find_spec
  File "<frozen importlib._bootstrap_external>", line 1532, in find_spec
  File "<frozen importlib._bootstrap_external>", line 1504, in _get_spec
  File "<frozen importlib._bootstrap_external>", line 1483, in _path_importer_cache
OSError: [Errno 107] Transport endpoint is not connected

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 2

In [ ]:
print(f"Path for data is: {data_dir}")

In [ ]:
#set the train and test paths

train_val_dir = data_dir / "train"  #contains both train and val data
test_dir = data_dir / "test"

print(f"Train & Val data path: {train_val_dir}")
print(f"Test data path: {test_dir}")

### 2.2 Checking data distribution

In [ ]:
#checking data distribution for train val data

import os

# total normal data count
normal_count = len(os.listdir(train_val_dir / "NORMAL"))

#total pneumonia data count
pneumonia_count = len(os.listdir(train_val_dir / "PNEUMONIA"))

#total count
total_data_count = normal_count + pneumonia_count

#percentage
normal_percentage = (normal_count / total_data_count) * 100
pneumonia_percentage = (pneumonia_count / total_data_count) * 100

print(f"Total normal data count: {normal_count} ({normal_percentage:.2f}%)")
print(f"Total pneumonia data count: {pneumonia_count} ({pneumonia_percentage:.2f}%)")
print(f"Total data count: {total_data_count}")

In [ ]:
import matplotlib.pyplot as plt

# Labels and sizes
labels = [f"NORMAL ({normal_count})", f"PNEUMONIA ({pneumonia_count})"]

# labels = ['NORMAL', 'PNEUMONIA']
sizes = [normal_percentage, pneumonia_percentage]
counts = [normal_count, pneumonia_count]

fig, axs = plt.subplots(1, 2, figsize=(10, 4))

# ---- Pie Chart ----
axs[0].pie(
    sizes,
    labels=labels,
    autopct='%1.2f%%',
    startangle=90,
    explode=(0.05, 0)
)
axs[0].set_title('Dataset Distribution (%)')
axs[0].axis('equal')

# ---- Bar Plot ----
axs[1].bar(labels, counts, color=["C0", "C1"])
axs[1].set_title('Dataset Distribution (Counts)')
axs[1].set_xlabel('Classes')
axs[1].set_ylabel('Number of Samples')

# Add a main title to the combined plot
fig.suptitle('Combined Dataset Distribution of Train and Val Datasets', fontsize=16)

plt.tight_layout()
plt.show()

- **The dataset is highly imbalanced dataset**. $N:P ≈ 1:3$
- Use of **weighted loss function** for training will be benifitial

### 2.3 Split the training data into train and val splits

$80 \% $ training and $20 \% $ validation split from train data. This is done inside the `src.dataloader.create_datloader()` function which returns the train, val, test dataloaders. (See pt. 3 below)

## 3. Create transforms for test and train data and create train, val and test dataloaders

- The transforms are created seperatly for train and test sets
    

In [ ]:
from src.transform import train_transforms, test_transforms
from src.dataloader import create_dataloaders

BATCH_SIZE = 32

train_dataloader, val_dataloader, test_dataloader, class_names = create_dataloaders(
    train_val_dir=train_val_dir,
    test_dir=test_dir,
    train_transform=train_transforms,
    test_transform=test_transforms,
    val_size=0.2,
    batch_size=BATCH_SIZE,
    num_workers=2
)

print(f"[INFO] Dataloaders created successfully with classes: {class_names}")

In [ ]:
print(f"Classes: {class_names}")
print(f"Train batches: {len(train_dataloader)} each of batch size {BATCH_SIZE}")
print(f"Val batches: {len(val_dataloader)} each of batch size {BATCH_SIZE}")
print(f"Test batches: {len(test_dataloader)} each of batch size {BATCH_SIZE}")

### 3.1 Visualize some random data from dataloaders

#### 3.1.1 Train dataloader

In [ ]:
from src.plots import plot_data

plot_data(dataloader=train_dataloader, class_names=class_names, k=10, title="10 Random Samples From Train dataloader")

#### 3.1.2 Val dataloader

In [ ]:
plot_data(dataloader=val_dataloader, class_names=class_names, k=10, title="10 Random Samples From Val dataloader")

#### 3.1.3 Test dataloader

In [ ]:
plot_data(dataloader=test_dataloader, class_names=class_names, k=10, title="10 Random Samples From Test dataloader")

## 4. Model building

4 models are choosen as each of them differ in application and architecture heavily from one another:
- Resnet 50: widely used baseline model
- DenseNet: Medically justified via CheXNet (Stanford's landmark chest X-ray paper, 2017)
- EffNet b2: Efficient architecture CNN model
- ViT-B/16: Transformer based modern architecture

All pretrained models are trained using transfer learning.
The Weights obtained from training on IMAGENET1K_V1 dataset are used for every model.

The model architecture can be found in the `src/models/architectures/`

## Modular scripts

### src/

In [ ]:
%%writefile src/dataset.py
"""
Contains:
    - function to download the data from kaggle
    - ChestXRayDataset class
"""

import shutil
import os
import kagglehub
from pathlib import Path

def download_data(target_dir: str) -> Path:
    """
    Download the data from Kaggle using kagglehub, moves the train, test data it to target_dir. Merge the val and train data and returns the path where train, test splits are placed.

    Args:
        - target_dir (str): local path where the data will be stored
    Returns:
        - Path: path where the train, test, val splits are placed
    """

    target_dir = Path(target_dir)

    # The final location where train, test, val splits will be placed
    final_data_path = target_dir / "chest_xray"

    #1. Check if the final organized data already exists
    if (final_data_path / "train").exists():
        print(f"[INFO] Dataset already exists at {final_data_path}. Skipping download...")
        return final_data_path

    # 2. Create target directory if it doesn't exist
    if not target_dir.exists():
        print(f"[INFO] Creating directory at {target_dir}...")
        target_dir.mkdir(parents=True, exist_ok=True)

    try:
        print("[INFO] Starting download from Kaggle via kagglehub...")
        # kagglehub downloads and unzipping into a global cache
        cache_path = Path(kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia"))

        # 3. Define the source of the nested mess in the cache
        # The structure in this specific zip is: chest_xray/chest_xray/[train, test, val]
        source_content_path = cache_path / "chest_xray" / "chest_xray"

        print(f"[INFO] Finished Downloading.")
        print(f"[INFO] Organizing and moving data to {final_data_path}...")

        # Create the final folder if it doesn't exist
        final_data_path.mkdir(parents=True, exist_ok=True)

        # 4. Move train, test folders from cache to target_dir. Merge train and val
        for split in ["train", "test"]:
            src = source_content_path / split
            dst = final_data_path / split

            if src.exists():
                # If the destination already exists: remove old data and move the new data
                if dst.exists():
                    shutil.rmtree(dst)

                # Copy from cache to dest
                shutil.copytree(src, dst)
                print(f"[INFO] Successfully moved {split} split.")

        # Merge val/NORMAL and val/PNEUMONIA into train/NORMAL and train/PNEUMONIA respectively
        val_src = source_content_path / "val"
        if val_src.exists():
            for class_dir in val_src.iterdir():          # NORMAL, PNEUMONIA
                if class_dir.is_dir():
                    train_class_dst = final_data_path / "train" / class_dir.name
                    train_class_dst.mkdir(parents=True, exist_ok=True)

                    for img_file in class_dir.iterdir():
                        dst_file = train_class_dst / img_file.name
                        # Rename on collision to avoid silent overwrites
                        if dst_file.exists():
                            dst_file = train_class_dst / f"val_{img_file.name}"

                        shutil.copy2(img_file, dst_file)

            print("[INFO] Successfully merged val and train splits.")


        print("[SUCCESS]  Dataset organized.")

    except Exception as e:
        print(f"[ERROR] An error occurred: {e}")
        return None

    print(f"[INFO] Data directory is ready at: {final_data_path}")
    return final_data_path

import torch
import numpy as np
from typing import Callable, Any
from PIL import Image
from .transform import basic_transform

class ChestXRayDataset(torch.utils.data.Dataset):
    """
    A Dataset wrapper that applies Albumentations transforms to an
    ImageFolder or Subset instance.

    NOTE:
        - If no transfom is given then applies ToTensorV2() transform and convert to tensor. But doesn't normalize or scale.
        - The base ImageFolder must be created with transform=None to avoid double-transform issues.
    """

    def __init__(self,
                 dataset: torch.utils.data.Dataset,
                 transform:  Callable[[Any], Any]=basic_transform) -> None:

        self.dataset = dataset
        self.transform = transform

        if isinstance(self.dataset, torch.utils.data.Subset):
            # dataset is an instance of Subset class
            self.samples = [self.dataset.dataset.samples[i] for i in self.dataset.indices]
            self.class_to_idx = self.dataset.dataset.class_to_idx
            self.classes = self.dataset.dataset.classes

        else:
            # dataset is an instance of Dataset class
            self.samples = dataset.samples
            self.classes = dataset.classes
            self.class_to_idx = dataset.class_to_idx

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        img_path, label = self.samples[index]

        # fetch image as PIL image and convert to np array (H, W, C)
        image = np.array(Image.open(img_path).convert("RGB"))

        # apply transform
        image = self.transform(image=image)["image"]    #return Tensor obj (C, H, W)

        return image, label


In [ ]:
%%writefile src/transform.py

"""
This file contains transforms for train and test data
"""

import albumentations as A
from albumentations.pytorch import ToTensorV2


IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
IMAGE_SIZE    = 224

# transform for test and validation set
test_transforms = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])


# transform for training set
train_transforms = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),

    # geometrical variations
    A.HorizontalFlip(p=0.4),
    A.Rotate(limit=10, p=0.5),  # ±10% rotation
    A.Affine(
        translate_percent=0.05,       # ±5% shift
        scale=(0.95, 1.05),           # ±5% zoom
        rotate=0,                     # 0 rotation (done above)
        p=0.4
    ),

    # simulate scanner condition & quality
    A.RandomBrightnessContrast(
        brightness_limit=0.2,
        contrast_limit=0.2,
        p=0.4
    ),

    # randomly remove rect. patches
    A.CoarseDropout(
        num_holes_range=(1, 8),
        hole_height_range=(8, 16),
        hole_width_range=(8, 16),
        fill=0,     # fill with black pixels
        p=0.3
    ),

    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),  #(input - mean) / std
    ToTensorV2(),
])

# basic transform to convert to tensor object
basic_transform = A.Compose([
    ToTensorV2()
])

In [ ]:
%%writefile src/dataloader.py
"""This file contains:
    - functions to create dataloaders
    - function to split train data into train and validation splits
    - function to calculate the weightage of pos and neg class in a dataloader
"""

import os
from typing import Callable, Tuple, List, Any, Optional

import torch
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from .dataset import ChestXRayDataset


def create_dataloaders(batch_size: int,
                       val_size: float,
                       num_workers: int,
                       persistent_workers: bool,
                       train_val_dir: Optional[str]=None,
                       test_dir: Optional[str]=None,
                       train_transform: Optional[Callable[[Any], Any]]=None,
                       test_transform: Optional[Callable[[Any], Any]]=None,
                       dataloader_type: str="all") -> Tuple[DataLoader, DataLoader, DataLoader, List[str]]:

    """
    1. Creates train and val splits subsets based on val_size
    2. Creates train, val and test datasets with given transforms
    3. Creates and returns train, val and test dataloaders with given batch size and num workers
    NOTE:
        - If dataloader_type="all" then train, val and test dataloaders are given
        - If dataloader_type="train" then only train and val dataloaders are given
        - If dataloader_type="test" then only test dataloader is given

    Args:
        - batch_size (int): batch size for the dataloaders
        - val_size (float): percentage of data to be used for validation set
        - num_workers (int): number of workers to be used for the dataloaders
        - persistent_workers (bool): whether to use persistent workers for the dataloaders
        - train_val_dir (str): path to the train and val data
        - test_dir (str): path to the test data
        - train_transform (Callable): transform to be applied to the train data
        - test_transform (Callable): transform to be applied to the test data
        - dataloader_type (str): type of dataloader to be created. Can be "all", "train" or "test"

    Returns:
        - train_dataloader, val_dataloader, test_dataloader (torch.utils.data.DataLoader): train, val and test dataloaders

    Raises:
        - ValueError: if dataloader_type is not in ["all", "train", "test"]
    """

    # validate dataloader_type
    if dataloader_type not in ["all", "train", "test"]:
        raise ValueError(
            f"dataloader_type must be in ['all', 'train', 'test']. ({dataloader_type} given)"
        )


    if dataloader_type in ["all", "train"]:

        # train_transform and train_val_dir is required
        if train_transform is None or train_val_dir is None:
            raise ValueError(
                f"train_transform and train_val_dir must be provided for {dataloader_type} dataloader_type"
            )

        #creating train and val splits and getting the train and val subsets
        train_subset, val_subset = create_test_val_split(
            train_val_dir=train_val_dir,
            val_size=val_size
        )

        #create the transformed dataset for train and val splits
        train_dataset = ChestXRayDataset(dataset=train_subset, transform=train_transform)
        val_dataset = ChestXRayDataset(dataset=val_subset, transform=test_transform)

        #creating train and val dataloaders
        train_dataloader = DataLoader(
            dataset=train_dataset,
            batch_size=batch_size,
            shuffle=True,
            num_workers=num_workers,
            persistent_workers=persistent_workers,
            pin_memory=True
        )

        val_dataloader = DataLoader(
            dataset=val_dataset,
            batch_size=batch_size,
            shuffle=False,
            num_workers=num_workers,
            persistent_workers=persistent_workers,
            pin_memory=True
        )

        class_names = train_dataset.classes


        # return only train dataloader if dataloader_type is 'train'
        if dataloader_type == "train":
            return train_dataloader, val_dataloader, class_names


    if dataloader_type in ["all", "test"]:

        # test_transform and test_val_dir is required
        if test_transform is None or test_dir is None:
            raise ValueError(
                f"test_transform and test_dir must be provided for {dataloader_type} dataloader_type"
            )

        #create test dataset
        test_dataset = ChestXRayDataset(dataset=ImageFolder(test_dir, transform=None), transform=test_transform)

        #create test dataloader
        test_dataloader = DataLoader(
            dataset=test_dataset,
            batch_size=batch_size,
            shuffle=False,
            num_workers=num_workers,
            persistent_workers=persistent_workers,
            pin_memory=True
        )

        class_names = test_dataset.classes

        # return only test dataloader if dataloader_type is 'test'
        if dataloader_type == "test":
            return test_dataloader, class_names

    return train_dataloader, val_dataloader, test_dataloader, class_names



from torch.utils.data import Subset
from sklearn.model_selection import train_test_split

def create_test_val_split(train_val_dir: str,
                          val_size: float) -> Tuple[Subset, Subset]:
    """
    Splits the data into training and validation split preserving the class balance
    Args:
        - train_val_dir (str): path to the train and val data
        - val_size (float): percentage of data to be used for validation

    Returns:
        - train_subset (Subset): train subset of the data
        - val_subset (Subset): validation subset of the data

    NOTE:
        - Here no transform is used as the train and val will require different transforms. Use the TransformedDataset class for creating a transfomed dataset.
    """

    # 1. Create the dataset for train and val data
    train_val_dataset = ImageFolder(root=train_val_dir, transform=None)

    # 2. Split into train and val splits
    targets = train_val_dataset.targets #get list of all targets

    train_idx, val_idx = train_test_split(
        range(len(train_val_dataset)),
        test_size=val_size,
        stratify=targets,   #ensure the class balance while spliting
        random_state=42
    )

    #get the train and val subsets
    train_subset = Subset(train_val_dataset, train_idx)
    val_subset = Subset(train_val_dataset, val_idx)

    return train_subset, val_subset


def get_class_weights(dataloader: DataLoader,
                      eps: float = 1e-6) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Computes pos and neg class weights from a dataloader in a single pass.

    Args:
        - dataloader (DataLoader): dataloader to compute weights from
        - eps (float): small value to avoid division by zero

    Returns:
        - tuple: (pos_weight, neg_weight) as float32 tensors
    """

    total_pos, total_neg = 0, 0

    for _, labels in dataloader:
        labels = labels.view(-1)
        total_pos += (labels == 1).sum().item()
        total_neg += (labels == 0).sum().item()

    total = total_pos + total_neg

    pos_weight = torch.tensor(total / (2 * total_pos + eps), dtype=torch.float32)
    neg_weight = torch.tensor(total / (2 * total_neg + eps), dtype=torch.float32)

    return pos_weight, neg_weight

In [ ]:
%%writefile src/utils.py
import torch
import random
import numpy as np

def set_seeds(seed: int=42)->None:
    """
    Sets seed across all libraries for full reproducibility.
    Args:
        - seed (int): seed value
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


from collections import Counter

def get_loader_distribution(loader: torch.utils.data.DataLoader):
    """
    Returns the distribution of data in the dataloader

    Args:
        - loader (torch.utils.data.DataLoader): dataloader to get the distribution of

    Returns:
        - dict
    """

    counter = Counter()

    for _, labels in loader:
        counter.update(labels.tolist())

    return counter


In [ ]:
%%writefile src/plots.py

import torch
import numpy as np
from .transform import IMAGENET_STD, IMAGENET_MEAN

def denormalize(tensor: torch.Tensor)->np.ndarray:
    """
    Denormalize a tensor of shape (C, H, W) and converts to numpy array of shape (H, W, C). Returns a numpy array suitable for plotting.

    Normalization: (input - mean) / std
    Denormalization: input = (output × std) + mean

    Args:
        - tensor (torch.Tensor): tensor to be denormalized

    Returns:
        - np.ndarray: denormalized tensor

    """

    #make the STD & MEAN tensor obj
    std = torch.tensor(IMAGENET_STD)
    mean = torch.tensor(IMAGENET_MEAN)

    #clone to avoide modifying original tensor
    image = tensor.clone()

    #denormalize: (image * std) + mean
    image = image * std.view(3, 1, 1) + mean.view(3, 1, 1)

    image = image.clamp(0, 1).permute(1, 2, 0).cpu().numpy()

    return image


import matplotlib.pyplot as plt
import random
from typing import List

def plot_data(dataloader: torch.utils.data.DataLoader,
              class_names: List[str],
              k: int=10,
              title: str="Random Sample From Images")->None:

    """
    Plots k samples for the given dataloader

    Args:
        - dataloader (torch.utils.data.DataLoader): dataloader to plot
        - k (int): number of samples to plot

    Raise:
        - ValueError: if k is not a multiple of 2 or greater than 10
    """

    # k must be multiple of 2 and should be <= 10
    if k % 2 != 0 or k > 10:
        raise ValueError("k must be a multiple of 2 and should be <= 10")

    img, labels = next(iter(dataloader))

    #get k random samples from the datasetloader
    random_idx = random.sample(range(len(img)), k)


    #plot the samples
    fig, axes = plt.subplots(nrows=2, ncols=k//2, figsize=(12, 7))
    axes = axes.flatten()  #flatten to 1D array

    for i, idx in enumerate(random_idx):

        axes[i].imshow(denormalize(img[idx]))   #denormalize before plotting

        axes[i].set_title(f"{class_names[labels[idx]]}")
        axes[i].axis("off")


    fig.suptitle(title, fontsize=15)
    plt.tight_layout()
    plt.show()



### models/

In [ ]:
%%writefile src/models/architectures/resnet50.py
"""
Contains functions to do the following for resnet50 model:
    - create_model: instantiate a resnet50 model for transfer learning with pretrained weights (trained on IMAGENET1K_V2 dataset). Freeze the backbone and keeps the classifier layer trainable.

    - create_optimizer: create optimizer for transfer learning or fine-tuning

    - unfreeze_for_finetune: unfreeze the last N residual blocks for fine tuning the model.
      Classifier layer is kept unfrozen. Keeps remaining layers frozen.

"""

import torch
from torchvision import models

# Ordered from last to first (unfreezing starts from the end)
_RESNET_BLOCKS = ["layer4", "layer3", "layer2", "layer1"]

_MAX_LAYERS = len(_RESNET_BLOCKS)

def _validate_n_layers(n_layers: int) -> None:
    """
    Validates that n_layers is within the acceptable range [1, _MAX_LAYERS].

    Args:
        - n_layers (int): number of residual blocks to unfreeze

    Raises:
        - ValueError: if n_layers is out of range
    """
    if n_layers < 1 or n_layers > _MAX_LAYERS:
        raise ValueError(
            f"n_layers must be between 1 and {_MAX_LAYERS} for ResNet50. "
            f"({n_layers} given)"
        )


def create_model(num_classes: int=2)->torch.nn.Module:
    """
    Creates a resnet50 model with pretrained weights (trained on IMAGENET1K_V2 dataset) and freezes the backbone.
    Makes the classifier layer trainable with num_classes output nodes.
    Useful for transfer learning.

    Args:
        - num_classes (int): number of output classes

    Returns:
        - model (torch.nn.Module): resnet50
    """

    # 1. Initialize a resnet50 model pretrained on IMAGENET1K_V2 dataset
    model_weights = models.ResNet50_Weights.DEFAULT
    model = models.resnet50(weights=model_weights)

    # 2. freeze the all layers
    for param in model.parameters():
        param.requires_grad = False

    # 3. Update the classifier (fc) layer to make last lasyer o/p as num_classes (trainable by default)
    model.fc = torch.nn.Linear(
        in_features=model.fc.in_features,
        out_features=num_classes,
        bias=True
    )

    return model


def create_optimizer(model: torch.nn.Module,
                     mode: str="transfer_learning",
                     n_layers: int=1,
                     tf_lr: float=1e-3,
                     ft_lr: float=1e-5,
                     lr_decay: float=0.1)->torch.optim.Optimizer:

    """
    Creates optimizer for transfer learning or fine-tuning.
    For transfer learning: only the fc layer parameters are updated.
    For fine-tuning: discriminative learning rates are applied across residual blocks.
    Each earlier block is scaled down by lr_decay:
        layer4  (position 0) -> ft_lr * (lr_decay ** 0) = ft_lr
        layer3  (position 1) -> ft_lr * (lr_decay ** 1)
        layer2  (position 2) -> ft_lr * (lr_decay ** 2)
        layer1  (position 3) -> ft_lr * (lr_decay ** 3)

    BN layers are internal to each residual block and are included automatically.
    The fc layer always uses tf_lr, independent of the decay chain.

    When n_layers=1, lr_decay has no effect (single block, no decay to apply).
    Args:
        - model (torch.nn.Module): model to be optimized
        - mode (str): "transfer_learning" or "fine_tuning"
        - tf_lr (float): learning rate for the fc layer (independent of decay chain)
        - ft_lr (float): base learning rate for the last unfrozen residual block (layer4)
        - n_layers (int): number of residual blocks to include in optimizer (fine_tuning only). Must be between 1 and 4.
        - lr_decay (float): multiplicative decay applied per block going deeper into the backbone. Must be in (0.0, 1.0]. Default 0.1. Typical values: 0.1 (aggressive), 0.3 (moderate).

    Returns:
        - optimizer (torch.optim.Optimizer)
    """

    # make sure mode is eigther transfer_learning or fine_tuning
    if mode not in ["transfer_learning", "fine_tuning"]:
        raise ValueError(
            f"Mode must be either 'transfer_learning' or 'fine_tuning'. ({mode} given)"
        )

    # Return optimizer for transfer_learning
    if mode=="transfer_learning":
        return torch.optim.Adam(params=model.fc.parameters(), lr=tf_lr)

    #for fine tuning
    _validate_n_layers(n_layers)

    param_groups = []

    # residual blocks params
    for i, residual_block in enumerate(_RESNET_BLOCKS[:n_layers]):
        block_lr = ft_lr * (lr_decay ** i)
        param_groups.append(
            {"params": getattr(model, residual_block).parameters(), "lr": block_lr}
        )

    return torch.optim.Adam(param_groups)



def unfreeze_for_finetune(model: torch.nn.Module,
                          n_layers: int=1)->None:
    """
    Unfreezes the last n_layers residual blocks of resent50.
    NOTE: the classifier layer (fc) is already unfrozen.

    Block unfreezing order (last to first):
        n_layers=1: layer4
        n_layers=2: layer4, layer3
        n_layers=3: layer4, layer3, layer2
        n_layers=4: layer4, layer3, layer2, layer1

    Args:
        - model (torch.nn.Module): resnet50 model
        - n_layers (int): number of residual blocks to unfreeze. Must be between 1 and 4.

    Raises:
        - ValueError: if n_layers is out of range [1, 4]
    """

    _validate_n_layers(n_layers)

    for residual_block in _RESNET_BLOCKS[:n_layers]:
        for param in getattr(model, residual_block).parameters():
            param.requires_grad = True

    #return is not required as nn.Module objs are mutable



In [ ]:
%%writefile src/models/architectures/densenet121.py
"""
Contains functions to do the following for densenet121 model:
    - create_model: instantiate a densenet121 model for transfer learning. Pretrained weights (trained on IMAGENET1K_V2 dataset). Freeze the backbone and keeps the classifier layer trainable.

    - create_optimizer: create optimizer for transfer learning or fine-tuning

    - unfreeze_for_finetune: unfreeze the last N dense blocks for fine tuning the model.
      Classifier layer is kept unfrozen. Keeps remaining layers frozen.

"""

import torch
from torchvision import models


# Ordered from last to first (unfreezing starts from the end)
# Each entry: (dense_block_attr, associated_norm_or_transition_attr)
_DENSENET_BLOCKS = [
    ("denseblock4", "norm5"),
    ("denseblock3", "transition3"),
    ("denseblock2", "transition2"),
    ("denseblock1", "transition1"),
]
_MAX_LAYERS = len(_DENSENET_BLOCKS)


def _validate_n_layers(n_layers: int) -> None:
    """
    Validates that n_layers is within the acceptable range [1, _MAX_LAYERS].

    Args:
        - n_layers (int): number of dense blocks to unfreeze

    Raises:
        - ValueError: if n_layers is out of range
    """
    if n_layers < 1 or n_layers > _MAX_LAYERS:
        raise ValueError(
            f"n_layers must be between 1 and {_MAX_LAYERS} for DenseNet121. "
            f"({n_layers} given)"
        )


def create_model(num_classes: int=2)->torch.nn.Module:
    """
    Creates a densenet121 model with pretrained weights (trained on IMAGENET1K_V2 dataset) and freeze the backbone.
    Makes the classifier layer trainable with num_classes output nodes.
    Usefull for transfer learning.

    Args:
        - num_classes (int): number of output classes

    Returns:
        - model (torch.nn.Module): densenet121

    """

    # 1. Initialize a densenet121 model pretrained on IMAGENET1K_V2 dataset
    model_weights = models.DenseNet121_Weights.DEFAULT
    model = models.densenet121(weights=model_weights)

    # 2. freeze the all layers
    for param in model.parameters():
        param.requires_grad = False

    # 3. Update the classifier layer to make last lasyer o/p as num_classes (trainable by default)
    model.classifier = torch.nn.Linear(
        in_features=model.classifier.in_features,
        out_features=num_classes,
        bias=True
    )

    return model


def create_optimizer(model: torch.nn.Module,
                     mode: str="transfer_learning",
                     n_layers: int = 1,
                     tf_lr: float=1e-3,
                     ft_lr: float=1e-5,
                     lr_decay: float=0.1)->torch.optim.Optimizer:

    """
    Creates optimizer for transfer learning or fine-tuning.
    For transfer learning: only the classifier layer parameters are updated.
    For fine-tuning: discriminative learning rates are applied across backbone blocks.
    The last (closest to classifier) block gets ft_lr, and each earlier block is scaled down by lr_decay:
        denseblock4  (position 0) -> ft_lr * (lr_decay ** 0) = ft_lr
        denseblock3  (position 1) -> ft_lr * (lr_decay ** 1)
        denseblock2  (position 2) -> ft_lr * (lr_decay ** 2)
        denseblock1  (position 3) -> ft_lr * (lr_decay ** 3)

    Associated norm/transition layers share the same LR as their dense block.
    The classifier always uses tf_lr, independent of the decay chain.

    When n_layers=1, lr_decay has no effect (single block, no decay to apply).

    Args:
        - model (torch.nn.Module): model to be optimized
        - mode (str): "transfer_learning" or "fine_tuning"
        - tf_lr (float): learning rate for the classifier layer (independent of decay chain)
        - ft_lr (float): base learning rate for the last unfrozen backbone block
        - n_layers (int): number of dense blocks to include in optimizer (fine_tuning only). Must be between 1 and 4.
        - lr_decay (float): multiplicative decay applied per block going deeper into the backbone. Must be in (0.0, 1.0]. Default 0.1. Typical values: 0.1 (aggressive), 0.3 (moderate).

    Returns:
        - optimizer (torch.optim.Optimizer)
    """

    if mode not in ["transfer_learning", "fine_tuning"]:
        raise ValueError(
            f"mode must be either 'transfer_learning' or 'fine_tuning'. ({mode} given)"
        )

    # Return optimizer for transfer_learning
    if mode=="transfer_learning":
        return torch.optim.Adam(params=model.classifier.parameters(), lr=tf_lr)


    #for fine tuning

    _validate_n_layers(n_layers)

    param_groups = []

    # Add last n_layers dense blocks and their associated norm/transition layers
    for i, (block_attr, associated_block_attr) in enumerate(_DENSENET_BLOCKS[:n_layers]):
        block_lr = ft_lr * (lr_decay ** i)
        param_groups.append(
            {"params": getattr(model.features, block_attr).parameters(), "lr": block_lr}
        )
        param_groups.append(
            {"params": getattr(model.features, associated_block_attr).parameters(), "lr": block_lr}
        )

    # include classifer head params
    param_groups.append(
        {"params": model.classifier.parameters(), "lr": tf_lr}
    )

    return torch.optim.Adam(param_groups)


def unfreeze_for_finetune(model: torch.nn.Module,
                          n_layers: int=1) -> None:
    """
    Unfreezes the last n_layers dense blocks and their associated norm/transition layers.
    NOTE: the classifier layer is already unfrozen.

    Block unfreezing order (last to first):
        n_layers=1: denseblock4 + norm5
        n_layers=2: denseblock4 + norm5, denseblock3 + transition3
        n_layers=3: above + denseblock2 + transition2
        n_layers=4: above + denseblock1 + transition1

    Args:
        - model (torch.nn.Module): densenet121 model
        - n_layers (int): number of dense blocks to unfreeze. Must be between 1 and 4.

    Raises:
        - ValueError: if n_layers is out of range [1, 4]
    """

    _validate_n_layers(n_layers)

    # Unfreeze last n_layers dense blocks and their associated norm/transition layers
    for block_attr, associated_block_attr in _DENSENET_BLOCKS[:n_layers]:

        # unfreeze dense block
        for param in getattr(model.features, block_attr).parameters():
            param.requires_grad = True

        # unfreeze associated block
        for param in getattr(model.features, associated_block_attr).parameters():
            param.requires_grad = True



In [ ]:
%%writefile src/models/architectures/efficientnet_b2.py
"""
Contains functions to do the following for efficientnet_b2 model:
    - create_model: instantiate a efficientnet_b2 model for transfer learning. Pretrained weights (trained on IMAGENET1K_V2 dataset). Freeze the backbone and keeps the classifier layer trainable.

    - create_optimizer: create optimizer for transfer learning or fine-tuning

    - unfreeze_for_finetune: unfreeze the last N MBConv blocks for fine tuning the model.
      features[8] (Conv2dNormActivation) is always unfrozen alongside any block. Classifier layer is kept unfrozen.
      Keeps remaining layers frozen.

"""

import torch
from torchvision import models

# Ordered from last to first (unfreezing starts from the end)
# features[8] is Conv2dNormActivation — always unfrozen for fine tuning, not counted in N
# N applies to features[7] down to features[0]
_EFFICIENTNET_BLOCKS = list(range(7, -1, -1))  # [7, 6, 5, 4, 3, 2, 1, 0]

_MAX_LAYERS = len(_EFFICIENTNET_BLOCKS)



def _validate_n_layers(n_layers: int) -> None:
    """
    Validates that n_layers is within the acceptable range [1, _MAX_LAYERS].

    Args:
        - n_layers (int): number of MBConv blocks to unfreeze

    Raises:
        - ValueError: if n_layers is out of range
    """
    if n_layers < 1 or n_layers > _MAX_LAYERS:
        raise ValueError(
            f"n_layers must be between 1 and {_MAX_LAYERS} for EfficientNet-B2. "
            f"({n_layers} given)"
        )


def create_model(num_classes: int=2)->torch.nn.Module:
    """
    Creates a efficientnet_b2 model with pretrained weights (trained on IMAGENET1K_V2 dataset) and freeze the backbone.
    Makes the classifier layer trainable with num_classes output nodes.
    Usefull for transfer learning.

    Args:
        - num_classes (int): number of output classes

    Returns:
        - model (torch.nn.Module): efficientnet_b2

    """

    # 1. Initialize a efficientnet_b2 model pretrained on IMAGENET1K_V2 dataset
    model_weights = models.EfficientNet_B2_Weights.DEFAULT
    model = models.efficientnet_b2(weights=model_weights)

    # 2. freeze the all layers
    for param in model.parameters():
        param.requires_grad = False

    # 3. Update the classifier layer to make last layer o/p as num_classes (trainable by default)
    model.classifier = torch.nn.Sequential(
        torch.nn.Dropout(p=0.3, inplace=True),
        torch.nn.Linear(
            in_features=model.classifier[1].in_features,
            out_features=num_classes,
            bias=True
        )
    )

    return model


def create_optimizer(model: torch.nn.Module,
                     mode: str="transfer_learning",
                     n_layers: int=1,
                     tf_lr: float=1e-3,
                     ft_lr: float=1e-5,
                     lr_decay: float=0.1)->torch.optim.Optimizer:

    """
    Creates optimizer for transfer learning or fine-tuning.
    For transfer learning: only the classifier layer parameters are updated.
    For fine-tuning: discriminative learning rates are applied across MBConv blocks.
    features[8] (Conv2dNormActivation) sits at position i=0 (same level as features[7]) and always gets ft_lr. Each earlier MBConv block is scaled down by lr_decay:
        features[8]  (position 0) -> ft_lr * (lr_decay ** 0) = ft_lr [always]
        features[7]  (position 0) -> ft_lr * (lr_decay ** 0) = ft_lr
        features[6]  (position 1) -> ft_lr * (lr_decay ** 1)
        features[5]  (position 2) -> ft_lr * (lr_decay ** 2)
        ...
        features[0]  (position 7) -> ft_lr * (lr_decay ** 7)

    The classifier always uses tf_lr, independent of the decay chain.

    When n_layers=1, lr_decay has no effect (single block, no decay to apply).

    Args:
        - model (torch.nn.Module): model to be optimized
        - mode (str): "transfer_learning" or "fine_tuning"
        - tf_lr (float): learning rate for the classifier layer (independent of decay chain)
        - ft_lr (float): base learning rate for the last unfrozen MBConv block (features[7])
        - n_layers (int): number of MBConv blocks to include in optimizer (fine_tuning only). Must be between 1 and 8.
        - lr_decay (float): multiplicative decay applied per block going deeper into the backbone. Must be in (0.0, 1.0]. Default 0.1. Typical values: 0.1 (aggressive), 0.3 (moderate).

    Returns:
        - optimizer (torch.optim.Optimizer)
    """

    # make sure mode is eigther transfer_learning or fine_tuning
    if mode not in ["transfer_learning", "fine_tuning"]:
        raise ValueError(
            f"mode must be either 'transfer_learning' or 'fine_tuning'. ({mode} given)"
        )

    # Return optimizer for transfer_learning
    if mode=="transfer_learning":
        return torch.optim.Adam(params=model.classifier.parameters(), lr=tf_lr)


    # for fine tuning

    _validate_n_layers(n_layers)

    param_groups = []

    for i, effnet_block_idx in enumerate(_EFFICIENTNET_BLOCKS[:n_layers]):
        block_lr = ft_lr * (lr_decay ** i)
        param_groups.append(
            {"params": model.features[effnet_block_idx].parameters(), "lr": block_lr}
        )

    # Conv2dNormActivation layer (features[8]) params
    param_groups.append(
        {"params": model.features[8].parameters(), "lr": ft_lr}
    )

    # classifier head params
    param_groups.append(
        {"params": model.classifier.parameters(), "lr": tf_lr}
    )

    return torch.optim.Adam(param_groups)


def unfreeze_for_finetune(model: torch.nn.Module,
                          n_layers: int=1)->None:

    """
    Unfreezes features[8] (Conv2dNormActivation) always, plus the last n_layers
    MBConv blocks (features[7] down to features[0]).
    NOTE: the classifier layer is already unfrozen.

    Block unfreezing order (last to first):
        n_layers=1: features[8] + features[7]
        n_layers=2: features[8] + features[7], features[6]
        ...
        n_layers=8: features[8] + features[7] → features[0]

    Args:
        - model (torch.nn.Module): efficientnet_b2 model
        - n_layers (int): number of MBConv blocks to unfreeze. Must be between 1 and 8.

    Raises:
        - ValueError: if n_layers is out of range [1, 8]
    """

    # unfreeze the last n_layers MBConv blocks
    for effnet_block_idx in _EFFICIENTNET_BLOCKS[:n_layers]:
        for param in model.features[effnet_block_idx].parameters():
            param.requires_grad = True

    # unfreeze the last Conv2dNormActivation layer (features[8])
    for param in model.features[8].parameters():
        param.requires_grad = True



In [ ]:
%%writefile src/models/architectures/vit_b_16.py
"""
Contains functions to do the following for vit_b_16 model:
    - create_model: instantiate a vit_b_16 model for transfer learning. Pretrained weights (trained on IMAGENET1K_V2 dataset). Freeze the backbone and keeps the classifier layer trainable.

    - create_optimizer: create optimizer for transfer learning or fine-tuning

    - unfreeze_for_finetune: unfreeze the last N encoder blocks for fine tuning the model.
      The ln layer is always unfrozen alongside any encoder block. Classifier layer is kept unfrozen.
      Keeps remaining layers frozen.

"""

import torch
from torchvision import models

# Ordered from last to first (unfreezing starts from the end)
# Total of 12 encoder layers in ViT-B/16
_ENCODER_BLOCKS = [f"encoder_layer_{i}" for i in range(11, -1, -1)]

_MAX_LAYERS = len(_ENCODER_BLOCKS)


def _validate_n_layers(n_layers: int) -> None:
    """
    Validates that n_layers is within the acceptable range [1, _MAX_LAYERS].

    Args:
        - n_layers (int): number of encoder blocks to unfreeze

    Raises:
        - ValueError: if n_layers is out of range
    """
    if n_layers < 1 or n_layers > _MAX_LAYERS:
        raise ValueError(
            f"n_layers must be between 1 and {_MAX_LAYERS} for ViT-B/16. "
            f"({n_layers} given)"
        )



def create_model(num_classes: int=2)->torch.nn.Module:
    """
    Creates a vit_b_16 model with pretrained weights (trained on IMAGENET1K_V2 dataset) and freezes the backbone.
    Makes the classifier layer trainable with num_classes output nodes.
    Useful for transfer learning.

    Args:
        - num_classes (int): number of output classes

    Returns:
        - model (torch.nn.Module): vit_b_16
    """


    # 1. Initialize a vit_b_16 model pretrained on IMAGENET1K_V2 dataset
    model_weights = models.ViT_B_16_Weights.DEFAULT
    model = models.vit_b_16(weights=model_weights)

    # 2. freeze the all layers
    for param in model.parameters():
        param.requires_grad = False

    # 3. Update the classifier head to make last layer o/p as num_classes (trainable by default)
    model.heads.head = torch.nn.Linear(
        in_features=model.heads.head.in_features,
        out_features=num_classes,
        bias=True
    )

    return model


def create_optimizer(model: torch.nn.Module,
                     mode: str="transfer_learning",
                     n_layers: int=1,
                     tf_lr: float=1e-4,
                     ft_lr: float=1e-5,
                     lr_decay: float=0.1)->torch.optim.Optimizer:

    """
    Creates optimizer for transfer learning or fine-tuning.
    For transfer learning: only the classifier head parameters are updated.
    For fine-tuning: discriminative learning rates are applied across encoder blocks.
    ln and encoder_layer_11 are treated as one logical unit at position i=0 (both get ft_lr).
    Each earlier encoder block is scaled down by lr_decay:

        ln                  (position 0) -> ft_lr * (lr_decay ** 0) = ft_lr
        encoder_layer_11    (position 0) -> ft_lr * (lr_decay ** 0) = ft_lr
        encoder_layer_10    (position 1) -> ft_lr * (lr_decay ** 1)
        encoder_layer_9     (position 2) -> ft_lr * (lr_decay ** 2)
        ...
        encoder_layer_0    (position 11) -> ft_lr * (lr_decay ** 11)

    When n_layers=1, lr_decay has no effect (single block, no decay to apply).

    The classifier head always uses tf_lr, independent of the decay chain.

    Args:
        - model (torch.nn.Module): model to be optimized
        - mode (str): "transfer_learning" or "fine_tuning"
        - tf_lr (float): learning rate for the classifier head (independent of decay chain)
        - ft_lr (float): base learning rate for the last unfrozen encoder block (encoder_layer_11)
        - n_layers (int): number of encoder blocks to include in optimizer (fine_tuning only). Must be between 1 and 12.
        - lr_decay (float): multiplicative decay applied per block going deeper into the encoder. Must be in (0.0, 1.0]. Default 0.1. Typical values: 0.1 (aggressive), 0.3 (moderate).
    Returns:
        - optimizer (torch.optim.Optimizer)
    """

    # make sure mode is eigther transfer_learning or fine_tuning
    if mode not in ["transfer_learning", "fine_tuning"]:
        raise ValueError(
            f"mode must be either 'transfer_learning' or 'fine_tuning'. ({mode} given)"
        )


    # Return optimizer for transfer_learning
    if mode=="transfer_learning":
        return torch.optim.Adam(params=model.heads.parameters(), lr=tf_lr)

    # for fine tuning

    _validate_n_layers(n_layers)


    param_groups = []

    # ecoder layer params
    for i, encoder_layer in enumerate(_ENCODER_BLOCKS[:n_layers]):
        block_lr = ft_lr * (lr_decay ** i)
        param_groups.append(
            {"params": getattr(model.encoder.layers, encoder_layer).parameters(), "lr": block_lr}
        )

    # include the ln layer
    param_groups.append(
        {"params": model.encoder.ln.parameters(), "lr": tf_lr}
    )

    # include the classifier head params
    param_groups.append(
        {"params": model.heads.parameters(), "lr": tf_lr}
    )

    return torch.optim.Adam(param_groups)



def unfreeze_for_finetune(model: torch.nn.Module,
                          n_layers: int=1)->None:
    """
    Unfreezes the last n_layers encoder blocks. The ln layer is always
    unfrozen whenever any encoder block is unfrozen.
    NOTE: the classifier head is already unfrozen.

    Block unfreezing order (last to first):
        n_layers=1 : encoder_layer_11 + ln
        n_layers=2 : encoder_layer_11, encoder_layer_10 + ln
        ...
        n_layers=12: encoder_layer_11 → encoder_layer_0 + ln

    Args:
        - model (torch.nn.Module): vit_b_16 model
        - n_layers (int): number of encoder blocks to unfreeze. Must be between 1 and 12.

    Raises:
        - ValueError: if n_layers is out of range [1, 12]
    """

    _validate_n_layers(n_layers)

    # unfreeze encoder_layers from end
    for encoder_layer in _ENCODER_BLOCKS[:n_layers]:
        for param in getattr(model.encoder.layers, encoder_layer).parameters():
            param.requires_grad = True

    # unfreeze ln
    for param in model.encoder.ln.parameters():
        param.requires_grad = True



In [ ]:
%%writefile src/models/models.py
"""
Contains registry of models and their architectures
"""

import torch
from .architectures import resnet50, densenet121, efficientnet_b2, vit_b_16

model_registry = {
    "resnet50": resnet50,
    "densenet121": densenet121,
    "efficientnet_b2": efficientnet_b2,
    "vit_b_16": vit_b_16
}


def _validate_model(model_name: str) -> None:
    """
    Validate whether a model is present in the model registry.

    Args:
        - model_name (str): name of the model to validate

    Raises:
        - ValueError: if model is not present in the registry
    """
    if model_name not in model_registry.keys():
        raise ValueError(
            f"Model name: {model_name} is not present in model registry. "
            f"Available models: {list(model_registry.keys())}"
        )


def create_model(model_name: str,
                 num_classes: int=1)->torch.nn.Module:

    """
    Creates a model from the registry.
    Note: By default the num_classes is set to 1 which means binary classification. For multiclass classification set it to a value greater than 2.

    Args:
        - model_name (str): name of the model to create
        - num_classes (int): number of classes for o/p node of classifier layer in model

    Returns:
        - model (torch.nn.Module): model created from the registry

    Raises:
        - ValueError: if num_classes is not greater than 2 for multiclass classification
    """

    _validate_model(model_name)

    # num_classes can be either 1 for binary classification or grater than 2 for multiclass classification
    if num_classes==2:
        raise ValueError(f"num_classes must be greater than 2 for multiclass classification ({num_classes} given)")

    return model_registry[model_name].create_model(num_classes=num_classes)


def create_optimizer(model_name: str,
                     model: torch.nn.Module,
                     mode: str="transfer_learning",
                     n_layers: int=1,
                     tf_lr: float=1e-3,
                     ft_lr: float=1e-5,
                     lr_decay: float=0.1)->torch.optim.Optimizer:
    """
    Returns optimizer for given model_name and model instance for model present in registry.

    For transfer learning (mode="transfer_learning"): only the classifier layer is updated.

    For fine-tuning (mode="fine_tuning"): discriminative learning rates are applied across the last n_layers backbone blocks. The last block gets ft_lr, and each earlier block is scaled by lr_decay_factor:
        last block       -> ft_lr
        second last      -> ft_lr * lr_decay_factor
        third last       -> ft_lr * lr_decay_factor^2
        ...

    The classifier always uses tf_lr, independent of the decay chain.
    When lr_decay_factor=1.0 (default).

    Args:
        - model_name (str): name of the model
        - model (torch.nn.Module): model to create optimizer for
        - mode (str): "transfer_learning" or "fine_tuning"
        - tf_lr (float): learning rate for the classifier layer (independent of decay chain)
        - ft_lr (float): base learning rate for the last unfrozen backbone block
        - n_layers (int): number of feature blocks to optimize (fine_tuning only).
            Valid range depends on architecture:
            - resnet50       : 1 to 4
            - densenet121    : 1 to 4
            - efficientnet_b2: 1 to 8
            - vit_b_16       : 1 to 12
        - lr_decay_factor (float): multiplicative decay per block going deeper into backbone. Must be in (0.0, 1.0]. Default 0.1. Typical values: 0.1 (aggressive), 0.3 (moderate).

    Returns:
        - optimizer (torch.optim.Optimizer): optimizer for the model
    """


    _validate_model(model_name)

    return model_registry[model_name].create_optimizer(
        model=model,
        mode=mode,
        n_layers=n_layers,
        tf_lr=tf_lr,
        ft_lr=ft_lr,
        lr_decay=lr_decay
    )


def unfreeze_for_finetune(model_name: str,
                          model: torch.nn.Module,
                          n_layers: int=1)->None:
    """
    Unfreezes the last n_layers feature blocks of the given model for fine-tuning.
    Associated norm/transition layers are unfrozen alongside their respective blocks.
    NOTE: the classifier layer is already unfrozen.

    Valid n_layers range per architecture:
        - resnet50       : 1 to 4  (layer4 → layer1)
        - densenet121    : 1 to 4  (denseblock4 → denseblock1)
        - efficientnet_b2: 1 to 8  (features[7] → features[0], features[8] always unfrozen)
        - vit_b_16       : 1 to 12 (encoder_layer_11 → encoder_layer_0, ln always unfrozen)

    Args:
        - model_name (str): name of the model
        - model (torch.nn.Module): model to be fine-tuned
        - n_layers (int): number of feature blocks to unfreeze (default: 1)

    Raises:
        - ValueError: if n_layers is out of range for the given architecture
    """

    _validate_model(model_name)

    model_registry[model_name].unfreeze_for_finetune(model, n_layers=n_layers)


### training loops

In [ ]:
%%writefile src/trainer/engine.py
"""
Contains functions to train and eval model over a single epoch.
Also a print function to display results in beautiful formatted way.
"""


import torch

from typing import Dict
from tqdm.auto import  tqdm

from torchmetrics.classification import (
    BinaryAccuracy,
    BinaryF1Score,
    BinaryPrecision,
    BinaryRecall,
    BinarySpecificity,
    BinaryAUROC
)

def train_step(model: torch.nn.Module,
                dataloader: torch.utils.data.DataLoader,
                loss_fn: torch.nn.Module,
                optimizer: torch.optim.Optimizer,
                device: str)->Dict:
    """
    Trains the given model over a single epoch

    Args:
        - model (torch.nn.Module): model to be trained
        - dataloader (torch.utils.data.DataLoader): dataloader for training data
        - loss_fn (torch.nn.Module): loss function for the model
        - optimizer (torch.optim.Optimizer): optimizer for the model
        - device (str): device to use for training

    Returns:
        - results (Dict): dictionary containing the training results for the epoch
    """

    metrics = {
        "recall": BinaryRecall(zero_division=0).to(device),
        "precision": BinaryPrecision(zero_division=0).to(device),
        "auroc": BinaryAUROC().to(device),
        "f1_score": BinaryF1Score(zero_division=0).to(device),
        "specificity": BinarySpecificity(zero_division=0).to(device),
        "accuracy": BinaryAccuracy(zero_division=0).to(device)
    }

    total_loss = 0.0

    model.to(device)
    model.train()

    pbar = tqdm(dataloader, desc="Train", leave=False)  #tqdm Bar

    for image, label in pbar:

        image, label = image.to(device), label.to(device)

        label_float = label.float()
        label_long = label.long()

        logits = model(image).squeeze(1)
        loss = loss_fn(logits, label_float)

        preds = (logits >= 0).long()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # metric updation
        total_loss += loss.item()
        for metric in metrics.values():
            metric.update(preds, label_long)

        # adding live metrics to tqdm bar (computed before)
        pbar.set_postfix({
            "auroc": f"{metrics['auroc'].compute().item():.4f}",
            "loss": f"{loss.item():.4f}"
        })

    # compute the metrics
    results = {k: v.compute().item() for k, v in metrics.items()}
    results["loss"] = total_loss / len(dataloader)

    # reset all metrics
    for metric in metrics.values():
        metric.reset()

    return results


def eval_step(model: torch.nn.Module,
              dataloader: torch.utils.data.DataLoader,
              loss_fn: torch.nn.Module,
              device: str)->Dict:
    """
    Evaluate the given model over a single epoch

    Args:
        - model (torch.nn.Module): model to be evaluated
        - dataloader (torch.utils.data.DataLoader): dataloader for evaluation data
        - loss_fn (torch.nn.Module): loss function for the model
        - device (str): device to use for evaluation

    Returns:
        - results (Dict): dictionary containing the evaluation results for the epoch
    """

    metrics = {
        "recall": BinaryRecall(zero_division=0).to(device),
        "precision": BinaryPrecision(zero_division=0).to(device),
        "auroc": BinaryAUROC().to(device),
        "f1_score": BinaryF1Score(zero_division=0).to(device),
        "specificity": BinarySpecificity(zero_division=0).to(device),
        "accuracy": BinaryAccuracy(zero_division=0).to(device)
    }

    total_loss = 0.0

    model.to(device)

    model.eval()
    with torch.inference_mode():

        pbar = tqdm(dataloader, desc="Eval ", leave=False)  #tqdm bar

        for image, label in pbar:

            image, label = image.to(device), label.to(device)

            label_float = label.float()
            label_long = label.long()

            logits = model(image).squeeze(1)
            loss = loss_fn(logits, label_float)

            preds = (logits >= 0).long()

            # update eval metrics
            total_loss += loss.item()
            for metric in metrics.values():
                metric.update(preds, label_long)

            # adding live metrics to tqdm bar (computed before)
            pbar.set_postfix({
                "auroc": f"{metrics['auroc'].compute().item():.4f}",
                "loss": f"{loss.item():.4f}"
            })


    # compute the metrics
    results = {k: v.compute().item() for k, v in metrics.items()}
    results["loss"] = total_loss / len(dataloader)

    # reset the metrics
    for metric in metrics.values():
        metric.reset()

    return results


def print_epoch_results(epoch: int,
                        epochs: int,
                        train_results: Dict,
                        eval_results: Dict)->None:
    """
    Display the train and test results of a single epoch in terminal in a formated way
    """

    print(f"Epoch [{epoch}/{epochs}]")
    print("Train ", " | ".join(f"{k}: {v:.4f}" for k, v in train_results.items()))
    print("Eval  ", " | ".join(f"{k}: {v:.4f}" for k, v in eval_results.items()))


In [ ]:
%%writefile src/trainer/train.py
"""
Contains the train function to train the model for given epochs
"""

import torch

from typing import Dict, Any, Optional, List
from tqdm.auto import  tqdm
from timeit import default_timer as timer
from pathlib import Path

import wandb

from .engine import train_step, eval_step, print_epoch_results


def train(train_dataloader: torch.utils.data.DataLoader,
          eval_dataloader: torch.utils.data.DataLoader,
          model: torch.nn.Module,
          loss_fn: torch.nn.Module,
          optimizer: torch.optim.Optimizer,
          model_name: str,
          run_name: str,
          config: Dict[str, Any],
          artifacts_dir: str,
          project_name: str,
          epochs: int,
          device: str,
          wandb_tags: Optional[List[str]]=None)->Dict:

    """
    Trains a model over given epochs with W&B experiment tracking and checkpointing.
    Note:
        - This training function is used for both future extraction and finetuning by setting the model, optimizer.
        - Each call to train() is one independent W&B run (finetune after future extraction are 2 differnt runs)

    Args:
        - train_dataloader  : dataloader for training data
        - eval_dataloader   : dataloader for evaluation/validation data
        - model             : model to be trained
        - loss_fn           : loss function
        - optimizer         : optimizer
        - model_name        : architecture name, e.g. "resnet50" (used in       checkpoint filename & W&B)
        - run_name          : unique run identifier, e.g. "resnet50-tl"
        - config            : dict of hyperparameters to log in W&B (lr, batch_size, epochs, mode, etc.)
        - artifacts_dir     : directory where the best checkpoint .pth will be saved. Eg: '/content/drive/MyDrive/Colab Notebooks/My Projects/Pneumonia Detection/artifacts/models'
        - project_name      : W&B project name (default: "pneumonia-detection")
        - epochs            : number of training epochs
        - device            : device to train on

    Returns:
        - results (Dict): train and eval metric dicts per epoch, plus path to best checkpoint
    """

    # W&B initializations
    wandb.init(
        project=project_name,
        name=run_name,
        tags=wandb_tags or [],
        config={
            "model_name": model_name,
            "epochs": epochs,
            "device": device,
            **config
        }
    )

    # create proper checkpoint path
    artifacts_dir = Path(artifacts_dir)
    artifacts_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = artifacts_dir / f"{run_name}.pth"

    # store the train and eval metric of each epoch
    results = {
        "train": [],
        "eval": []
    }

    best_auc = 0.0   # checkpoint creation tracker for best model

    #tqdm bar
    pbar = tqdm(range(1, epochs+1))

    start_time = timer()

    for epoch in pbar:
        pbar.set_description(f"Epoch [{epoch}/{epochs}]")

        train_results = train_step(
            model=model,
            loss_fn=loss_fn,
            optimizer=optimizer,
            dataloader=train_dataloader,
            device=device
        )
        eval_results = eval_step(
            model=model,
            loss_fn=loss_fn,
            dataloader=eval_dataloader,
            device=device
        )

        # display the results
        print_epoch_results(epoch, epochs, train_results, eval_results)

        # store the results
        results["train"].append(train_results)
        results["eval"].append(eval_results)

        # log metrics group by train and eval
        wandb.log({
            "epoch": epoch,
            **{f"train/{k}": v for k, v in train_results.items()},
            **{f"eval/{k}": v for k, v in eval_results.items()}
        })

        # model checkpoint. Save model if beats current best f1 score
        current_auc = eval_results["auroc"]
        if current_auc > best_auc and eval_results["recall"] > 0.80:
            best_auc = current_auc

            #save the model
            torch.save(
                obj={
                    "epoch": epoch,
                    "model_name": model_name,
                    "run_name": run_name,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "best_auc": best_auc
                }, f=checkpoint_path
            )

            print(f"[INFO] New checkpoint with eval AU-ROC score={best_auc:.4f} saved at {checkpoint_path}\n")


    end_time = timer()
    total_time = end_time - start_time
    print(f"Total training time: {total_time:.3f} seconds (~{round(total_time/60, 2)} minutes)\n")

    # save model as W&B artifacts
    model_artifact = wandb.Artifact(name=run_name, type="model")
    model_artifact.add_file(str(checkpoint_path))
    wandb.log_artifact(model_artifact)

    wandb.finish()

    #store checkpoint path, best f1, total training time to results
    results["checkpoint_path"] = str(checkpoint_path)
    results["best_auc"] = best_auc
    results["time_sec"] = total_time

    return results



In [ ]:
%%writefile src/trainer/runner.py
"""
Contains the main entry point for training the model

run_experiment()
│
├── VALIDATION
│   ├── ValueError  → load_checkpoint AND ft_epochs both given
│   ├── ValueError → dataloaders AND dataloader params both given
│   └── ValueError  → mode not in ["transfer_learning", "fine_tuning"]
│
├── DATALOADER RESOLUTION
│   └── created using: create_dataloaders()
│       └── pos_weight: set as hyperparam
│
├── run_name AUTO-CONSTRUCTION
│
├── CASE A: mode="transfer_learning", ft_epochs=None, load_checkpoint=None
│   ├── create_model → create_optimizer(mode="transfer_learning")
│   ├── train(epochs)
│   │   └── W&B Run:
│   │       ├── run_name: resnet50-TL_LR1e-3-EP8-B32
│   │       └── tags: [model_name, "TL"]
│   └── return tl_results
│
├── CASE B: mode="transfer_learning", ft_epochs given, load_checkpoint=None
│   ├── Phase 1 (Transfer Learning):
│   │   ├── create_model → create_optimizer(mode="transfer_learning")
│   │   ├── train(epochs)
│   │   │   └── W&B Run 1:
│   │   │       ├── run_name: resnet50-TL_LR1e-3-EP8-B32
│   │   │       └── tags: [model_name, "TL", "TL_FT"]
│   │
│   ├── unfreeze_for_finetune(n_layers)
│   ├── create_optimizer(mode="fine_tuning", lr=ft_lr)
│   │
│   ├── Phase 2 (Fine-Tuning):
│   │   ├── train(ft_epochs)
│   │   │   └── W&B Run 2:
│   │   │       ├── run_name:
|   |   |            resnet50-TL_LR1e-3-EP8-B32__FT_LR1e-5-EP5-N_LY2
│   │   │       ├── tags: [model_name, "FT", "TL_FT"]
│   │   │       └── config:
│   │   │           ├── tl_run_name
│   │   │           └── tl_checkpoint_path
│   │
│   └── return {"tl": tl_results, "ft": ft_results}
│
├── CASE C: mode="fine_tuning", load_checkpoint given, ft_epochs given
│   ├── create_model
│   ├── load state_dict from checkpoint
│   ├── unfreeze_for_finetune(n_layers)
│   ├── create_optimizer(mode="fine_tuning", lr=ft_lr)
│   ├── train(ft_epochs)
│   │   └── W&B Run:
│   │       ├── run_name:
|   |            resnet50-TL_LR1e-3-EP8-B32__FT_LR1e-5-EP5-N_LY2_CP
│   │       ├── tags: [model_name, "FT", "TL_FT", "finetune_checkpoint-tf"]
│   │       └── config:
│   │           ├── tl_run_name (from checkpoint)
│   │           └── tl_checkpoint_path
│   └── return ft_results
│
└── CASE D: mode="fine_tuning", load_checkpoint=None, ft_epochs given
    ├── create_model
    ├── unfreeze_for_finetune(n_layers)
    ├── create_optimizer(mode="fine_tuning", lr=ft_lr)
    ├── train(ft_epochs)
    │   └── W&B Run:
    │       ├── run_name: resnet50-FT_LR1e-5-EP5-N_LY2
    │       └── tags: [model_name, "FT"]
    └── return ft_results
"""


import os
import warnings
from pathlib import Path
from typing import Callable, Dict, Any, Optional, Tuple, Union

import torch

from .train import train
from ..transform import train_transforms, test_transforms
from ..dataloader import create_dataloaders, get_class_weights
from ..models.models import create_model, create_optimizer, unfreeze_for_finetune
from ..utils import set_seeds

set_seeds()


#NOTE: later update this constants from .yaml file.
_project_name = "pneumonia-detection"

_train_val_dir = "/content/drive/MyDrive/Colab Notebooks/My Projects/Pneumonia Detection/data/chest_xray/train"
_artifacts_dir = "/content/drive/MyDrive/Colab Notebooks/My Projects/Pneumonia Detection/artifacts/models"

_batch_size = 32
_val_size = 0.2
_epochs = 5

_device = "cuda" if torch.cuda.is_available() else "cpu"
_num_workers = os.cpu_count()

_tf_lr = 1e-3
_ft_lr = 1e-5
_lr_decay = 0.2
_n_layers = 1

_pos_weight = 0.673


def run_experiment(model_name: str,
                   mode: str="transfer_learning",

                   # W&B
                   project_name: str=_project_name,
                   extra_config: Optional[Dict[str, Any]]=None,

                   # dataloaders: gets auto created if not passed directly
                   train_val_dir: str=_train_val_dir,
                   train_transform: Callable[[Any], Any]=train_transforms,
                   test_transform: Callable[[Any], Any]=test_transforms,
                   batch_size: int=_batch_size,
                   val_size: float=_val_size,
                   num_workers: int=_num_workers,
                   persistent_workers: Optional[bool]=None,

                   # loss fn
                   pos_weight: float=_pos_weight,

                   # training
                   epochs: int=_epochs,
                   artifacts_dir: str=_artifacts_dir,
                   device: str=_device,

                   # optimizer/model architecture
                   tf_lr: float=_tf_lr,
                   ft_lr: float=_ft_lr,
                   lr_decay: float=_lr_decay,
                   n_layers: int=_n_layers,

                   # fine tune control
                   ft_epochs: Optional[int]=None,
                   load_checkpoint: Optional[str]=None):

    """
    Top-level experiment entry point.
    1. Validate all input parameters
    2. Resolve dataloaders (direct or auto-created)
    3. Resolve pos_weight (provided or computed)
    4. Build the loss function
    5. Route to _run_transfer_learning() or _run_fine_tuning()

    ── Cases ───────────────────────────────────────────────────────────
    Case A │ mode="transfer_learning", ft_epochs=None
           │ Feature extraction only. Returns tl_results.
           │
    Case B │ mode="transfer_learning", ft_epochs=<int>
           │ Feature extraction → fine-tuning.
           │ Returns {"tl": tl_results, "ft": ft_results}.
           │
    Case C │ mode="fine_tuning", load_checkpoint=<path>, ft_epochs=<int>
           │ Fine-tuning from existing checkpoint.
           │ Returns ft_results.
           │
    Case D │ mode="fine_tuning", load_checkpoint=None, ft_epochs=<int>
           │ Fine-tuning only on a freshly created model.
           │ Returns ft_results.

    Args:
        - model_name: name of the model to be used. Eg: resent50 (must be present in model_registry)
        - mode: "transfer_learning" or "fine_tuning"

        - project_name: W&B project name
        - extra_config: extra config to be passed to W&B

        - epochs: number of epochs to train for furture extraction
        - artifacts_dir: directory where the best checkpoint .pth are saved
        - device: device to train on

        - tf_lr: transfer learning learning rate
        - ft_lr: fine tuning learning rate
        - lr_decay: learning rate decay factor
        - n_layers: number of layers to unfreeze for fine tuning

        - ft_epochs: number of epochs to train for fine tuning
        - load_checkpoint: path to the checkpoint to load for fine tuning. Eg: '/content/drive/MyDrive/Colab Notebooks/My Projects/Pneumonia Detection/artifacts/models/resnet50-TL_LR1e-3-EP8-B32'

    Raises:
        - ValueError : mode is not "transfer_learning" or "fine_tuning"
        - ValueError : mode="transfer_learning" and load_checkpoint is given
        - ValueError : mode="fine_tuning" and ft_epochs = None
        - ValueError : load_checkpoint given and model_name does not match saved model
    """

    # Resolve persistent_wrokers
    if persistent_workers is None:
        persistent_workers = True if num_workers > 0 else False

    # Set the start method for multiprocessing. This is crucial for DataLoaders with num_workers > 0 in Colab.
    if persistent_workers:
        torch.multiprocessing.set_start_method('spawn', force=True)


    artifacts_dir = Path(artifacts_dir)
    train_val_dir = Path(train_val_dir)
    load_checkpoint = Path(load_checkpoint) if load_checkpoint else None

    # VALIDATE PARAMS---------------------

    # 1. Invalid mode
    if mode not in ("transfer_learning", "fine_tuning"):
        raise ValueError(
            f"`mode` must be 'transfer_learning' or 'fine_tuning' ('{mode}' given)."
        )

    # 2. load_checkpoint is meaningless for transfer learning
    if mode == "transfer_learning" and load_checkpoint is not None:
        raise ValueError(
            "`load_checkpoint` cannot be used with mode='transfer_learning'. "
            "To fine-tune an existing checkpoint, use mode='fine_tuning'."
        )

    # 3. fine_tuning requires at least one of load_checkpoint or ft_epochs
    if mode == "fine_tuning" and ft_epochs is None:
        raise ValueError(
            "mode='fine_tuning' requires `ft_epochs` to be set.\n"
            "  - For fine-tuning from a checkpoint (Case C): set both ft_epochs and load_checkpoint.\n"
            "  - For fine-tuning a freshly created model (Case D): set `ft_epochs`."
        )

    # DATALOADER RESOLUTION-------------------
    train_dl, val_dl = _resolve_dataloaders(
        train_val_dir=train_val_dir,
        train_transform=train_transform,
        test_transform=test_transform,
        batch_size=batch_size,
        val_size=val_size,
        num_workers=num_workers,
        persistent_workers=persistent_workers
    )

    # POS WEIGHT RESOLUTION---------------------
    pos_weight_resolved = torch.tensor(pos_weight, device=device, dtype=torch.float32)

    # LOSS FUNCTION---------------------
    loss_fn = _loss_fn(name="binary_ce", pos_weight=pos_weight_resolved)

    # Case A and B
    # feature extraction --> fine tune if ft_epochs given
    if mode == "transfer_learning":

        # 1. Create a model
        model = create_model(model_name)

        # 2. Create an optimizer for model
        tf_optimizer = create_optimizer(
            model_name=model_name,
            model=model,
            mode="transfer_learning",
            tf_lr=tf_lr,
        )

        return _run_transfer_learning(
            model_name=model_name,
            model=model,
            train_dataloader=train_dl,
            val_dataloader=val_dl,
            tf_optimizer=tf_optimizer,
            loss_fn=loss_fn,
            epochs=epochs,
            batch_size=batch_size,
            tf_lr=tf_lr,
            ft_lr=ft_lr,
            lr_decay=lr_decay,
            n_layers=n_layers,
            ft_epochs=ft_epochs,
            extra_config=extra_config,
            artifacts_dir=artifacts_dir,
            project_name=project_name,
            device=device
        )

    # Case C and D
    # fine tuning from existing checkpoint or standalone fine tune

    if load_checkpoint is not None:

        # Case C: finetune from existing checkpoint

        checkpoint = torch.load(load_checkpoint, map_location=device, weights_only=True)

        tl_run_name   = checkpoint.get("run_name",   "unknown-tl-run")
        tl_model_name = checkpoint.get("model_name", model_name)

        # 4. model name mismatch guard
        if tl_model_name != model_name:
            raise ValueError(
                f"Model name mismatch: given '{model_name}' but the checkpoint at '{load_checkpoint}' was saved from '{tl_model_name}'"
            )

        model = create_model(model_name)
        model.load_state_dict(checkpoint["model_state_dict"])
        print(f"\n[INFO] Loaded checkpoint from '{load_checkpoint}'")
        print(f"[INFO] Parent TL run : '{tl_run_name}' "
              f"(epoch {checkpoint['epoch']}, AU-ROC: {checkpoint['best_auc']:.4f})")

        tl_checkpoint_path = load_checkpoint
        from_checkpoint    = True

    else:

        # Case D: standalone finetune
        model              = create_model(model_name)
        tl_run_name        = None
        tl_checkpoint_path = None
        from_checkpoint    = False

    # unfreeze backbone blocks (Cases C and D both)
    unfreeze_for_finetune(model_name, model, n_layers)
    print(f"[INFO] Unfroze last {n_layers} backbone block(s) for fine-tuning.")

    ft_optimizer = create_optimizer(
        model_name=model_name,
        model=model,
        mode="fine_tuning",
        n_layers=n_layers,
        tf_lr=tf_lr,
        ft_lr=ft_lr,
        lr_decay=lr_decay
    )

    return _run_fine_tuning(
        model_name=model_name,
        model=model,
        train_dataloader=train_dl,
        val_dataloader=val_dl,
        ft_optimizer=ft_optimizer,
        loss_fn=loss_fn,
        ft_epochs=ft_epochs,
        tf_lr=tf_lr,
        ft_lr=ft_lr,
        n_layers=n_layers,
        batch_size=batch_size,
        tl_run_name=tl_run_name,
        tl_checkpoint_path=tl_checkpoint_path,
        from_checkpoint=from_checkpoint,
        extra_config=extra_config,
        artifacts_dir=artifacts_dir,
        project_name=project_name,
        device=device
    )



def _run_transfer_learning(model_name: str,
                           model: torch.nn.Module,
                           train_dataloader: torch.utils.data.DataLoader,
                           val_dataloader: torch.utils.data.DataLoader,
                           tf_optimizer: torch.optim.Optimizer,
                           loss_fn: torch.nn.Module,
                           epochs: int,
                           batch_size: int,
                           tf_lr: float,
                           ft_lr: Optional[float],
                           lr_decay: Optional[float],
                           n_layers: Optional[int],
                           ft_epochs: Optional[int],
                           extra_config: Optional[Dict[str, Any]],
                           artifacts_dir: str,
                           project_name: str,
                           device: str) -> Dict:

    """
    Handles Transfer Learning — Case A and Case B.

    Case A (ft_epochs=None):
        Trains for `epochs` and returns tl_results directly.

    Case B (ft_epochs=<int>):
        Trains for `epochs`, reloads best TL checkpoint, unfreezes
        `n_layers` backbone blocks, re-creates optimizer with
        discriminative LRs, then delegates to _run_fine_tuning().
        Returns {"tl": tl_results, "ft": ft_results}.
    """

    # WandB run name
    tl_run_name = _build_tl_run_name(
        model_name=model_name,
        epochs=epochs,
        tf_lr=tf_lr,
        batch_size=batch_size
    )

    # pre build FT run name to add in TF config if TF is followed by FT
    ft_run_name = None
    if ft_epochs is not None:
        ft_run_name = tl_run_name + _build_ft_suffix(ft_lr, ft_epochs, n_layers)

    # wandb config
    tl_config = _build_config(
        model_name=model_name,
        mode="transfer_learning",
        epochs=epochs,
        batch_size=batch_size,
        tf_lr=tf_lr,
        ft_lr=ft_lr if ft_epochs else None,
        n_layers=n_layers if ft_epochs else None,
        ft_epochs=ft_epochs,
        load_checkpoint=None,
        extra=extra_config
    )

    tl_config["phase"] = "TL"
    tl_config["has_ft_continualtion"] = ft_epochs is not None

    if ft_run_name is not None:
        tl_config["ft_run_name"] = ft_run_name

    # WandB tags
    tl_tags = [model_name, "TL"]
    if ft_epochs is not None:
        tl_tags.append("TL_FT")

    print(f"\n{'-'*70}")

    print(f"PHASE — Transfer Learning  |  {model_name}")
    print(f"Run: {tl_run_name}")
    print(f"Epochs: {epochs}")

    print(f"{'-'*70}\n")

    # TL training
    tl_results = train(
        train_dataloader=train_dataloader,
        eval_dataloader=val_dataloader,
        model=model,
        loss_fn=loss_fn,
        optimizer=tf_optimizer,
        model_name=model_name,
        run_name=tl_run_name,
        config=tl_config,
        artifacts_dir=artifacts_dir,
        project_name=project_name,
        epochs=epochs,
        device=device,
        wandb_tags=tl_tags
    )

    # Case A: return tl_results (only feature learning)
    if ft_epochs is None:
        return tl_results

    # Case B: prepare model and delegate to _run_fine_tuning
    tl_checkpoint_path = tl_results["checkpoint_path"]

    # reload best TL weights
    checkpoint = torch.load(tl_checkpoint_path, map_location=device, weights_only=True)
    model.load_state_dict(checkpoint["model_state_dict"])
    print(f"\n[INFO] Reloaded best TL weights from '{tl_checkpoint_path}' "
          f"(epoch {checkpoint['epoch']}, AU-ROC: {checkpoint['best_auc']:.4f})")

    # unfreeze last n_layers backbone blocks
    unfreeze_for_finetune(model_name, model, n_layers)
    print(f"[INFO] Unfroze last {n_layers} backbone block(s) for fine-tuning.")

    # re-create optimizer for fine tuning
    ft_optimizer = create_optimizer(
        model_name=model_name,
        model=model,
        mode="fine_tuning",
        n_layers=n_layers,
        tf_lr=tf_lr,
        ft_lr=ft_lr,
        lr_decay=lr_decay
    )

    ft_results = _run_fine_tuning(
        model_name=model_name,
        model=model,
        train_dataloader=train_dataloader,
        val_dataloader=val_dataloader,
        ft_optimizer=ft_optimizer,
        loss_fn=loss_fn,
        ft_epochs=ft_epochs,
        tf_lr=tf_lr,
        ft_lr=ft_lr,
        n_layers=n_layers,
        batch_size=batch_size,
        tl_run_name=tl_run_name, # for run name construction, layer 1 link
        tl_checkpoint_path=tl_checkpoint_path, # layer 3 link
        from_checkpoint=False,   # Case B — continuation, not checkpoint load
        extra_config=extra_config,
        artifacts_dir=artifacts_dir,
        project_name=project_name,
        device=device
    )

    return {"tl": tl_results, "ft": ft_results}


def _run_fine_tuning(model_name: str,
                     model: torch.nn.Module,
                     train_dataloader: torch.utils.data.DataLoader,
                     val_dataloader: torch.utils.data.DataLoader,
                     ft_optimizer: torch.optim.Optimizer,
                     loss_fn: torch.nn.Module,
                     ft_epochs: int,
                     tf_lr: float,
                     ft_lr: float,
                     n_layers: int,
                     batch_size: int,
                     tl_run_name: Optional[str],
                     tl_checkpoint_path: Optional[str],
                     from_checkpoint: bool,
                     extra_config: Optional[Dict[str, Any]],
                     artifacts_dir: str,
                     project_name: str,
                     device: str) -> Dict:

    """
    Handles Fine-Tuning — Cases B (continuation), C (from checkpoint), D (standalone).

    - Always receives a fully prepared model — instantiated, weighted,
    and unfrozen.
    - Never loads checkpoints, unfreezes layers, or creates optimizers.

    Run name logic:
        Case B/C (tl_run_name given):
            tl_run_name + __FT_LR=...-EP=...-N_LY=...
            Case C additionally appends _CP to denote checkpoint origin.
        Case D (tl_run_name is None):
            resnet50-FT_LR=1e-5-EP=5-N_LY=2

    W&B Config traceability:
        Layer 1 (config) : tl_run_name stored in ft_config
        Layer 2 (tag)    : "TL_FT" tag when linked to a TL run
        Layer 3 (config) : tl_checkpoint_path stored in ft_config

    Args:
        - from_checkpoint (bool): True for Case C (loaded from saved .pth),
                                  False for Case B (continuation) and Case D.
                                  Controls the _CP suffix and tags.
    """


    # FT run name
    if tl_run_name is not None:
        # Cases B and C — FT name references TL run name
        ft_run_name = tl_run_name + _build_ft_suffix(ft_lr, ft_epochs, n_layers)
        if from_checkpoint:
            ft_run_name += "_CP"   # suffix denotes this run loaded from a saved file
    else:
        # Case D — standalone FT, no TL parent
        ft_run_name = _build_standalone_ft_run_name(
            model_name=model_name,
            ft_lr=ft_lr,
            ft_epochs=ft_epochs,
            n_layers=n_layers
        )

    # WandB config
    ft_config = _build_config(
        model_name=model_name,
        mode="fine_tuning",
        epochs=ft_epochs,
        batch_size=batch_size,
        tf_lr=tf_lr,
        ft_lr=ft_lr,
        n_layers=n_layers,
        ft_epochs=ft_epochs,
        load_checkpoint=tl_checkpoint_path,
        extra=extra_config
    )
    ft_config["phase"]              = "FT"
    ft_config["tl_run_name"]        = tl_run_name        or "none"  # Layer 1
    ft_config["tl_checkpoint_path"] = tl_checkpoint_path or "none"  # Layer 3

    # WandB tags
    ft_tags = [model_name, "FT"]
    if tl_run_name is not None:
        ft_tags.append("TL_FT")
    if from_checkpoint:
        ft_tags.append("finetune_checkpoint-tf")

    print(f"\n{'-'*70}")

    print(f"PHASE — Fine-Tuning  |  {model_name}")
    print(f"Run  : {ft_run_name}")
    print(f"Epochs: {ft_epochs}")
    if tl_run_name:
        print(f"Parent TL run  : {tl_run_name}")
    if tl_checkpoint_path:
        print(f"From checkpoint: {tl_checkpoint_path}")

    print(f"{'-'*70}\n")

    ft_results = train(
        train_dataloader=train_dataloader,
        eval_dataloader=val_dataloader,
        model=model,
        loss_fn=loss_fn,
        optimizer=ft_optimizer,
        model_name=model_name,
        run_name=ft_run_name,
        config=ft_config,
        artifacts_dir=artifacts_dir,
        project_name=project_name,
        epochs=ft_epochs,
        device=device,
        wandb_tags=ft_tags
    )

    return ft_results


def _resolve_dataloaders(train_val_dir: str,
                         train_transform: Callable[[Any], Any],
                         test_transform: Callable[[Any], Any],
                         batch_size: int,
                         val_size: float,
                         num_workers: int,
                         persistent_workers: bool)->Tuple[torch.utils.data.DataLoader, torch.utils.data.DataLoader]:
    """
    Creates train and val dataloaders using create_dataloaders()
    Returns (train_dataloader, val_dataloader).
    """

    # only params passed
    train_dl, val_dl, _ = create_dataloaders(
        train_val_dir=train_val_dir,
        train_transform=train_transform,
        test_transform=test_transform,
        val_size=val_size,
        batch_size=batch_size,
        num_workers=num_workers,
        persistent_workers=persistent_workers,
        dataloader_type="train"
    )

    return train_dl, val_dl


def _fmt_lr(lr: float) -> str:
    """Formats 0.001 → '1e-3', 0.0001 → '1e-4', 0.1 → '1e-1'."""

    return f"{lr:.0e}".replace("e-0", "e-").replace("e+0", "e")


def _build_tl_run_name(model_name: str,
                       epochs: int,
                       tf_lr: float,
                       batch_size: int) -> str:
    """
    Builds run name for the Transfer Learning phase.
    Used as-is for Case A, and as the base for Case B FT name.

    Example:
        resnet50-TL_LR1e-3-EP8-B32
    """
    return (
        f"{model_name.lower()}-TL"
        f"_LR{_fmt_lr(tf_lr)}-EP{epochs}-B{batch_size}"
    )


def _build_ft_suffix(ft_lr: float,
                     ft_epochs: int,
                     n_layers: int) -> str:
    """
    Builds the fine-tuning suffix appended to a TL run name.
    Used for Cases B and C.

    Example:
        __FT_LR1e-5-EP5-N_LY2
    """
    return f"__FT_LR{_fmt_lr(ft_lr)}-EP{ft_epochs}-N_LY{n_layers}"


def _build_standalone_ft_run_name(model_name: str,
                                  ft_lr: float,
                                  ft_epochs: int,
                                  n_layers: int) -> str:
    """
    Builds run name for Case D — fine-tuning with no prior TL phase.

    Example:
        resnet50-FT_LR1e-5-EP5-N_LY2
    """
    return (
        f"{model_name.lower()}-FT"
        f"_LR{_fmt_lr(ft_lr)}-EP{ft_epochs}-N_LY{n_layers}"
    )


def _build_config(model_name: str,
                  mode: str,
                  epochs: int,
                  batch_size: int,
                  tf_lr: float,
                  ft_lr: Optional[float],
                  n_layers: Optional[int],
                  ft_epochs: Optional[int],
                  load_checkpoint: Optional[str],
                  extra: Optional[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Builds the base W&B config dict capturing all hyperparameters.
    Additional phase-specific keys are added by each _run_* function.
    """

    config = {
        "model_name":       model_name,
        "mode":             mode,
        "epochs":           epochs,
        "batch_size":       batch_size,
        "tf_lr":            tf_lr,
        "ft_lr":            ft_lr,
        "n_layers":         n_layers,
        "ft_epochs":        ft_epochs,
        "load_checkpoint":  load_checkpoint,
    }
    if extra:
        config.update(extra)
    return config


def _loss_fn(name="binary_ce",
             pos_weight: Optional[float]=None):
    """
    Creates and returns the loss function based on the given name.

    Raises:
        - ValueError if name is not a supported loss function.
    """

    if name == "binary_ce":
        return torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    raise ValueError(f"Unknown loss function: '{name}'. Available: ['binary_ce']")


In [ ]:
import importlib
import src.dataloader
import src.trainer.runner
import src.trainer.engine
import src.trainer.train

# Reload the src.dataloader module to ensure the latest changes are loaded
importlib.reload(src.dataloader)
importlib.reload(src.trainer.runner)
importlib.reload(src.trainer.engine)
importlib.reload(src.trainer.train)

from src.trainer.runner import run_experiment

In [ ]:
import os

run_experiment(
    model_name="efficientnet_b2",
    num_workers=0,
    persistent_workers=False,

)

In [ ]:
pos_weight_val, neg_weight_val = get_class_weights(val_dataloader)
print(f"pos_weight: {pos_weight_val}, neg_weight: {neg_weight_val}")

In [ ]:
from src.models.models import  create_model, create_optimizer

resnet_50_test = create_model("resnet50")
loss_fn_weighted = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))
resnet50_optimizer = create_optimizer(
    model_name="resnet50",
    model=resnet_50_test,
    mode="transfer_learning",
    n_layers=1
)


In [ ]:
from src.trainer.train import train

resnet50_results = train(
    train_dataloader=train_dataloader,
    eval_dataloader=test_dataloader,
    model=resnet_50_test,
    loss_fn=loss_fn_weighted,
    optimizer=resnet50_optimizer,
    epochs=3
)


In [ ]:
import importlib
import src.trainer.engine
import src.trainer.train

# Force reload the modules to reflect the latest changes
importlib.reload(src.trainer.engine)
importlib.reload(src.trainer.train)

print("src.trainer.engine and src.trainer.train modules reloaded successfully.")

In [ ]:
densenet121_test = create_model("densenet121")
loss_fn_weighted = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))
densenet121_optimizer = create_optimizer(
    model_name="densenet121",
    model=densenet121_test,
    mode="transfer_learning",
    n_layers=1
)


densenet121_results = train(
    train_dataloader=train_dataloader,
    eval_dataloader=test_dataloader,
    model=densenet121_test,
    loss_fn=loss_fn_weighted,
    optimizer=densenet121_optimizer,
    epochs=3
)

In [ ]:
efficientnet_b2_test = create_model("efficientnet_b2")
loss_fn_weighted = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))
efficientnet_b2_optimizer = create_optimizer(
    model_name="efficientnet_b2",
    model=efficientnet_b2_test,
    mode="transfer_learning",
    n_layers=1
)


efficientnet_b2_results = train(
    train_dataloader=train_dataloader,
    eval_dataloader=test_dataloader,
    model=efficientnet_b2_test,
    loss_fn=loss_fn_weighted,
    optimizer=efficientnet_b2_optimizer,
    epochs=3
)

In [ ]:
from src.trainer.train import train

vit_b_16_test = create_model("vit_b_16")
loss_fn_weighted = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))
vit_b_16_optimizer = create_optimizer(
    model_name="vit_b_16",
    model=vit_b_16_test,
    mode="transfer_learning",
    n_layers=1
)


vit_b_16_results = train(
    train_dataloader=train_dataloader,
    eval_dataloader=test_dataloader,
    model=vit_b_16_test,
    loss_fn=loss_fn_weighted,
    optimizer=vit_b_16_optimizer,
    epochs=3
)

In [ ]:
import importlib
import src.models.models
import src.models.architectures
import src.models.architectures.densenet121 # Explicitly import the submodule

# Reload specific architecture submodule first
importlib.reload(src.models.architectures.densenet121)
# Then reload the models module which uses this architecture
importlib.reload(src.models.models)

from torchinfo import summary
from src.models.models import create_model, unfreeze_for_finetune, create_optimizer

densenet_model = create_model("densenet121")

unfreeze_for_finetune("densenet121", densenet_model)

create_optimizer(
    model_name="densenet121",
    model=densenet_model,
    mode="fine_tuning",
    n_layers=3
)

summary(
    model=densenet_model,
    input_size=(1, 3, 224, 224),
    col_names=["input_size", "output_size", "num_params", "trainable"],
    col_width=20,
    row_settings=["var_names"]
)